In [0]:
%pip install --force-reinstall databricks-automl-runtime==0.2.20.13
%pip install gluonts[torch]==0.15.1
%pip install mlflow==2.20.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
  Using cached databricks_automl_runtime-0.2.20.13-py2.py3-none-any.whl (56 kB)
  Attempting uninstall: databricks-automl-runtime
    Found existing installation: databricks-automl-runtime 0.2.20
    Not uninstalling databricks-automl-runtime at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-66d4ffe4-9529-4c00-9179-e938d114fee7
    Can't uninstall 'databricks-automl-runtime'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2

# DeepAR training
- This is an auto-generated notebook.
- To reproduce these results, attach this notebook to Serverless compute, and rerun it.
- Compare trials in the [MLflow experiment](#mlflow/experiments/3257488039771013).
- Clone this notebook into your project folder by selecting **File > Clone** in the notebook toolbar.

In [0]:
import mlflow
import databricks.automl_runtime

target_col = "value"
time_col = "timestamp"
unit = "day"

frequency_quantity = 1


split_col = "split_col"


horizon = 14

## Load Data

In [0]:
import mlflow
import os
import uuid
import shutil
import pandas as pd
import pyspark.pandas as ps

# Create temp directory to download input data from MLflow
input_temp_dir = os.path.join(os.environ["SPARK_LOCAL_DIRS"], "tmp", str(uuid.uuid4())[:8])
os.makedirs(input_temp_dir)

# Download the artifact and read it into a pandas DataFrame
input_data_path = mlflow.artifacts.download_artifacts(run_id="9cdd178b23bf48208756e2d28205e006", artifact_path="data", dst_path=input_temp_dir)

input_file_path = os.path.join(input_data_path, "training_data")
df_loaded = ps.from_pandas(pd.read_parquet(input_file_path))

# Preview data
# display(df_loaded.head(5))

## Aggregate data by `time_col` and `split_col`
Group the data by `time_col` and `split_col`, and take average if there are multiple `target_col` values in the same group.

In [0]:
group_cols = [time_col]
group_cols = group_cols + [split_col]




df_aggregated = df_loaded \
  .groupby(group_cols) \
  .agg(y=(target_col, 'avg')) \
  .reset_index()

# display(df_aggregated.head(5))

## Train DeepAR model
- Log relevant metrics to MLflow to track runs
- All the runs are logged under [this MLflow experiment](#mlflow/experiments/3257488039771013)
- Change the model parameters and re-run the training cell to log a different trial to the MLflow experiment

In [0]:
from databricks.automl_runtime.forecast import OFFSET_ALIAS_MAP
unit = OFFSET_ALIAS_MAP[unit]

In [0]:
def fill_na(df):
  for key in df:
    entry = df[key]
    if entry.isnull().values.any():
      df[key] = df[key].fillna(method='ffill')
      df[key] = df[key].fillna(method='bfill')
  return df

def deepar_data_prep(df):
  from gluonts.dataset.pandas import PandasDataset
  from databricks.automl_runtime.forecast.deepar.utils import set_index_and_fill_missing_time_steps

  df_full = set_index_and_fill_missing_time_steps(
    df,
    time_col=time_col,
    frequency_unit=unit,
    frequency_quantity=frequency_quantity,
  )
  df_train_val = set_index_and_fill_missing_time_steps(
    df[df[split_col].isin(['train', 'validate'])],
    time_col=time_col,
    frequency_unit=unit,
    frequency_quantity=frequency_quantity,
  )
  df_train = set_index_and_fill_missing_time_steps(
    df[df[split_col] == 'train'],
    time_col=time_col,
    frequency_unit=unit,
    frequency_quantity=frequency_quantity,
  )

  df_full = fill_na(df_full)
  df_train_val = fill_na(df_train_val)
  df_train = fill_na(df_train)

  ds = PandasDataset(df_full, target="y")
  ds_train_val = PandasDataset(df_train_val, target="y")
  ds_train = PandasDataset(df_train, target="y")

  return ds, ds_train_val, ds_train

In [0]:
spark.conf.set("spark.databricks.mlflow.trackHyperopt.enabled", "false")

In [0]:
import pandas as pd

In [0]:
from gluonts.torch import DeepAREstimator
from gluonts.torch.distributions import StudentTOutput, NegativeBinomialOutput
from gluonts.evaluation import make_evaluation_predictions
from gluonts.evaluation import Evaluator
from lightning.pytorch.callbacks import EarlyStopping

from functools import partial
from hyperopt import hp, fmin, SparkTrials, space_eval
import hyperopt

2026-01-30 12:18:57.156025: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 12:19:00.503532: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [0]:
mlflow_sample_input = df_loaded.tail(5).to_pandas()
mlflow_sample_input.drop(columns=[split_col], inplace=True)


split_prefix = "val_"

metric_short_to_full_name = {
  "smape": "smape",
  "rmse": "root_mean_squared_error",
  "mse": "mean_squared_error",
  "mae": "mean_absolute_error",
}
logged_primary_metric = split_prefix + metric_short_to_full_name["smape"]

def deepar_training(params, unit, frequency_quantity, horizon, train_data, early_stop_data, eval_data, eval_split):
  import mlflow

  max_epochs = params["max_epochs"]
  output_distribution = params.get("output_distribution", StudentTOutput())
  context_length = max(params["context_length"], 10)
  batch_size = params["batch_size"]

  with mlflow.start_run(experiment_id="3257488039771013"):
    mlflow.set_tag("estimator_name", "DeepAR")
    mlflow.set_tag("eval_split", eval_split)

    mlflow.log_param("max_epochs", max_epochs)
    mlflow.log_param("output_distribution", output_distribution.__class__.__name__.replace("Output", ""))
    mlflow.log_param("context_length", context_length)
    mlflow.log_param("batch_size", batch_size)

    deepar_estim = DeepAREstimator(freq=f"{frequency_quantity}{unit}",
                                   prediction_length=horizon,
                                   context_length=context_length,
                                   batch_size=batch_size,
                                   distr_output=output_distribution,
                                   num_feat_dynamic_real=0,
                                   trainer_kwargs={
                                     "max_epochs" : max_epochs,
                                     "callbacks": [EarlyStopping(monitor="val_loss", patience=10)],
                                     "enable_progress_bar": False,
                                     "enable_model_summary": False,
                                     "logger": False,
                                     "default_root_dir": "/tmp/checkpoints"
                                   })
    deepar_predictor = deepar_estim.train(train_data, validation_data=early_stop_data)
    forecast_it, ts_it = make_evaluation_predictions(dataset=eval_data, predictor=deepar_predictor, num_samples=100)
    evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
    deepar_results, _ = evaluator(list(ts_it), list(forecast_it))

    # Log metrics to mlflow
    metric_name_map = {"MSE": "mean_squared_error", "RMSE": "root_mean_squared_error", "MAPE": "mean_absolute_percentage_error", "sMAPE": "smape"}
    deepar_metrics = {metric_name_map[key]: deepar_results[key] for key in metric_name_map.keys() if key in deepar_results}
    deepar_metrics["mean_absolute_error"] = deepar_results["abs_error"] / (horizon * len(train_data))
    avg_metrics = pd.DataFrame(list(deepar_metrics.items()), columns=["index", "mean_metrics"])
    avg_metrics["index"] = split_prefix + avg_metrics["index"].astype(str)
    avg_metrics.set_index("index", inplace=True)
    avg_metrics_dict = avg_metrics.to_dict()["mean_metrics"]
    mlflow.log_metrics(avg_metrics_dict)

    # Save the model to mlflow
    
    deepar_model = DeepARModel(
      deepar_predictor,
      horizon=horizon,
      frequency_unit=unit,
      frequency_quantity=frequency_quantity,
      num_samples=100,
      target_col=target_col,
      time_col=time_col,
    )

    mlflow_deepar_log_model(deepar_model, sample_input=mlflow_sample_input)

  return {"loss": avg_metrics_dict[logged_primary_metric], "status": hyperopt.STATUS_OK}

In [0]:
import numpy as np

def deepar_tune_hyperparameters(ds, ds_train_val, ds_train):
  objective = partial(
    deepar_training,
    unit=unit,
    frequency_quantity=frequency_quantity,
    horizon=horizon,
    train_data=ds_train,
    early_stop_data=ds_train_val,
    eval_data=ds_train_val,
    eval_split="val",
  )

  search_space = {
    "batch_size": hp.choice("batch_size", [32, 64]),
    "context_length": hp.choice("context_length", [horizon, 2*horizon]),
    "max_epochs": hp.choice("max_epochs", [10, 20, 100]),
  }

  is_target_col_positive_ints = False
  if is_target_col_positive_ints:
    search_space.update({
      "output_distribution": hp.choice("output_distribution", [StudentTOutput(), NegativeBinomialOutput()]),
    })

  trials = SparkTrials()

  fmin(
    fn=objective,
    space=search_space,
    algo=hyperopt.tpe.suggest,
    max_evals=4,
    trials=trials,
    timeout=7200,
    rstate=np.random.default_rng(752673323),
    show_progressbar=False,
  )

In [0]:
import mlflow
from databricks.automl_runtime.forecast.deepar.model import DeepARModel, mlflow_deepar_log_model

ds, ds_train_val, ds_train = deepar_data_prep(df_aggregated.to_pandas())

import logging
_logger_hyperopt = logging.getLogger("hyperopt-spark")
_logger_hyperopt.setLevel(logging.ERROR)
deepar_tune_hyperparameters(ds=ds, ds_train_val=ds_train_val, ds_train=ds_train)

In [0]:
filter_string = "tags.estimator_name='DeepAR' AND tags.eval_split='val'"
runs_df = runs = mlflow.search_runs(experiment_ids=["3257488039771013"], filter_string=filter_string, order_by=[f"metrics.{logged_primary_metric} asc"])
best_run_id = runs_df["run_id"][0]
mlflow_run = mlflow.get_run(best_run_id)

# Display all DeepAR runs sorted by the metric
runs_df.loc[:, ["run_id", f"metrics.{logged_primary_metric}", "params.max_epochs", "params.output_distribution", "params.batch_size", "params.context_length"]].head()

,run_id,metrics.val_smape,params.max_epochs,params.output_distribution,params.batch_size,params.context_length
0,2b88ed6f9d1a4bd5a736a7a59b0c411c,0.258150,100,StudentT,64,28
1,d60557c1db4b42debd9ebffdb7ee2b21,0.351464,10,StudentT,32,28
2,d6788d02b159459dbd610b27f6ccf4bb,0.370380,100,StudentT,32,14
3,2aba797e382e4f86947480ea9ef947af,0.428316,100,StudentT,32,28
